# Api para obtener dataset partidos 

In [4]:
import requests
from datetime import datetime

In [ ]:


# Tu API key de API-Football
api_key = "12ae61a006ce6dbdcfb286e80b593923"
url = "https://v3.football.api-sports.io/fixtures"
headers = {"x-apisports-key": api_key}

# Definimos los dos equipos de Monterrey
equipos = {
    "Tigres UANL": 2279,
    "Monterrey (Rayados)": 2282
}

# Set para registrar los IDs de los partidos y evitar duplicar los Clásicos Regios
partidos_procesados = set()

print(f"{'FECHA':<12} | {'PARTIDO':<45} | {'RESULTADO':<10} | {'TORNEO / FASE'}")
print("-" * 115)

for nombre_equipo, team_id in equipos.items():
    # Hacemos la petición para el rango de años que necesitas
    querystring = {
        "team": str(team_id),
        "from": "2016-01-01",
        "to": "2023-12-31"
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()
    
    if 'response' in data:
        for partido in data['response']:
            fixture_id = partido['fixture']['id']
            
            # Si el partido ya se guardó (por ejemplo, un Tigres vs Rayados), lo saltamos
            if fixture_id in partidos_procesados:
                continue
            partidos_procesados.add(fixture_id)
            
            # 1. Fecha (YYYY-MM-DD)
            fecha_raw = partido['fixture']['date']
            fecha = datetime.strptime(fecha_raw[:10], "%Y-%m-%d").strftime("%Y-%m-%d")
            
            # 2. Partido (Local vs Visitante)
            local = partido['teams']['home']['name']
            visitante = partido['teams']['away']['name']
            encuentro = f"{local} vs {visitante}"
            
            # 3. Resultado
            goles_local = partido['goals']['home']
            goles_visit = partido['goals']['away']
            
            # Validamos que el partido realmente se haya jugado y tenga goles
            if goles_local is not None and goles_visit is not None:
                resultado = f"{goles_local} - {goles_visit}"
            else:
                resultado = "No Disp."
            
            # 4. Torneo y Fase
            torneo = partido['league']['name']
            fase = partido['league']['round']
            competicion = f"{torneo} ({fase})"
            
            print(f"{fecha:<12} | {encuentro:<45} | {resultado:<10} | {competicion}")

FECHA        | PARTIDO                                       | RESULTADO  | TORNEO / FASE
-------------------------------------------------------------------------------------------------------------------
